In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
from pathlib import Path
import sys

# Project root
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

#from modules import helpers
#from modules import metrics
#from modules import read_timeseries as read
from modules import MOHIDHDFtoNetcdf as conv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import glob
import h5py

## Calibration against SST data

In [10]:
filename = r'C:\Users\karoa\MOHID_internship\MOHID_model_workflow\data\calibration\WaterProperties_test.hdf5'
outdir   = r'C:\Users\karoa\MOHID_internship\MOHID_model_workflow\data\calibration\\'   

t, dates = conv.MOHIDHdf5toNetcdf(filename, outdir=outdir)

print(f"Wrote {t} timesteps")

Wrote 49 timesteps


In [11]:
# concat files with time
files = sorted(glob.glob(r'C:\Users\karoa\MOHID_internship\MOHID_model_workflow\data\calibration\WaterProperties_*.nc'))
datasets = [xr.open_dataset(f) for f in files]

# Stack them along a 'time' dimension
ds = xr.concat(datasets, dim='time')

# Close the underlying file handles now that data is in memory
for d in datasets:
    d.close()

In [12]:
# assign coordinate according to dates
with h5py.File(filename, 'r') as f:
    time_keys = sorted(f['Time'].keys())
    real_dates = []
    for k in time_keys:
        arr = f['Time'][k][:].ravel().astype(float)   # 6 numbers: y, m, d, H, M, S
        y, m, d, H, M, S = arr[:6]
        real_dates.append(pd.Timestamp(int(y), int(m), int(d),
                                       int(H), int(M), int(round(S))))

ds = ds.assign_coords(time=('time', real_dates))

In [13]:
ds

<xarray.Dataset> Size: 2MB
Dimensions:      (time: 49, lat: 101, lon: 25)
Coordinates:
  * time         (time) datetime64[us] 392B 2023-01-13 ... 2023-01-15
  * lat          (lat) float64 808B 28.23 28.23 28.23 ... 28.16 28.16 28.16
  * lon          (lon) float64 200B -16.84 -16.84 -16.84 ... -16.86 -16.86
Data variables:
    salinity     (time, lat, lon) float32 495kB -9.9e+15 -9.9e+15 ... 36.0 36.0
    temperature  (time, lat, lon) float32 495kB -9.9e+15 -9.9e+15 ... 21.0 21.0
    Bathymetry   (time, lat, lon) float64 990kB nan nan nan ... 47.37 47.37